In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# BTCUSDT Daily OHLCV — Machine Learning Prediction System

## Project Objective

This project analyzes historical BTCUSDT daily OHLCV data and develops machine
learning models to predict the next-day Bitcoin return and market direction.

### Primary Task
Predict the next-day BTCUSDT return:

\[
Return_{t+1} = \frac{Close_{t+1}}{Close_t} - 1
\]

### Secondary Task
Predict whether the next day's closing price will increase:

- `1` → Next-day Close > Current-day Close
- `0` → Next-day Close <= Current-day Close

### Project Workflow

1. Dataset Loading
2. Data Understanding
3. Data Validation & Cleaning
4. Exploratory Data Analysis
5. Feature Engineering
6. Target Creation
7. Chronological Train/Validation/Test Split
8. Baseline Models
9. Machine Learning Models
10. Model Evaluation
11. Feature Importance
12. Final Model Selection
13. Next-Day Prediction
14. Conclusion & Limitations

> This project is for educational and experimental purposes only.
> It is NOT financial advice and does not guarantee future market performance.

# 1. Import Libraries

In [ ]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingRegressor,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

from sklearn.inspection import permutation_importance

import joblib

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

# 2. Load Dataset

In [ ]:
# ==========================================
# FIND BTCUSDT CSV DATASET
# ==========================================

import glob

csv_files = glob.glob("/kaggle/input/datasets/trevorfrench/btcusdt-daily-ohlcv-data/btc_usdt_daily.csv", recursive=True)

print("CSV files found:")
for file in csv_files:
    print(file)

# Find the BTCUSDT dataset
btc_files = [
    file for file in csv_files
    if "btc" in file.lower() or "usdt" in file.lower()
]

if len(btc_files) == 0:
    raise FileNotFoundError(
        "BTCUSDT CSV not found. Check that the Kaggle dataset is attached."
    )

DATA_PATH = btc_files[0]

print("\nSelected dataset:")
print(DATA_PATH)

# Load dataset
df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

# 3. Dataset Understanding

In [ ]:
# ==========================================
# BASIC DATASET INFORMATION
# ==========================================

print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
display(df.dtypes.to_frame("Data Type"))

print("\nFirst 5 Rows:")
display(df.head())

print("\nLast 5 Rows:")
display(df.tail())

print("\nDataset Information:")
df.info()

In [ ]:
# ==========================================
# DESCRIPTIVE STATISTICS
# ==========================================

display(df.describe().T)

In [ ]:
# ==========================================
# DATA QUALITY CHECK
# ==========================================

print("Missing Values:")
missing_values = df.isnull().sum()

missing_table = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": (missing_values / len(df)) * 100
})

display(missing_table)

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nTotal Missing Values:")
print(df.isnull().sum().sum())

## 3.1 Timestamp Processing

In [ ]:
# ==========================================
# TIMESTAMP PROCESSING
# ==========================================

# Convert Unix timestamp to datetime
df["datetime"] = pd.to_datetime(df["id"], unit="s")

# Sort chronologically
df = df.sort_values("datetime").reset_index(drop=True)

print("Date Range:")
print("Start:", df["datetime"].min())
print("End:", df["datetime"].max())

print("\nFirst records after sorting:")
display(df.head())

print("\nLast records after sorting:")
display(df.tail())

## 3.2 Data Validation

In [ ]:
# ==========================================
# OHLC CONSISTENCY CHECK
# ==========================================

# High should be >= Open, Close, Low
invalid_high = (
    (df["high"] < df["open"]) |
    (df["high"] < df["close"]) |
    (df["high"] < df["low"])
)

# Low should be <= Open, Close, High
invalid_low = (
    (df["low"] > df["open"]) |
    (df["low"] > df["close"]) |
    (df["low"] > df["high"])
)

print("Invalid High Records:", invalid_high.sum())
print("Invalid Low Records:", invalid_low.sum())

if invalid_high.sum() == 0 and invalid_low.sum() == 0:
    print("\nOHLC consistency check passed.")
else:
    print("\nWarning: Some OHLC records are inconsistent.")

In [ ]:
# ==========================================
# BTCUSDT CLOSING PRICE
# ==========================================

plt.figure(figsize=(16, 6))

plt.plot(
    df["datetime"],
    df["close"],
    linewidth=1.5
)

plt.title("BTCUSDT Daily Closing Price")
plt.xlabel("Date")
plt.ylabel("Closing Price (USDT)")
plt.grid(alpha=0.3)

plt.show()

# 4. Exploratory Data Analysis

In [ ]:
# ==========================================
# OHLC PRICE VISUALIZATION
# ==========================================

plt.figure(figsize=(16, 7))

plt.plot(df["datetime"], df["open"], label="Open")
plt.plot(df["datetime"], df["high"], label="High")
plt.plot(df["datetime"], df["low"], label="Low")
plt.plot(df["datetime"], df["close"], label="Close")

plt.title("BTCUSDT Daily OHLC Prices")
plt.xlabel("Date")
plt.ylabel("Price (USDT)")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# DAILY RETURNS
# ==========================================

df["return_1d"] = df["close"].pct_change()

display(df[["datetime", "close", "return_1d"]].head(10))

print("Return Statistics:")
display(df["return_1d"].describe())

In [ ]:
# ==========================================
# RETURN DISTRIBUTION
# ==========================================

plt.figure(figsize=(12, 6))

sns.histplot(
    df["return_1d"].dropna(),
    bins=80,
    kde=True
)

plt.title("Distribution of Daily BTCUSDT Returns")
plt.xlabel("Daily Return")
plt.ylabel("Frequency")

plt.grid(alpha=0.2)
plt.show()

In [ ]:
# ==========================================
# DAILY RETURN TIME SERIES
# ==========================================

plt.figure(figsize=(16, 6))

plt.plot(
    df["datetime"],
    df["return_1d"],
    linewidth=0.8
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1
)

plt.title("BTCUSDT Daily Returns")
plt.xlabel("Date")
plt.ylabel("Daily Return")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# VOLUME AND MARKET ACTIVITY
# ==========================================

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

axes[0].plot(
    df["datetime"],
    df["vol"],
    linewidth=1
)

axes[0].set_title("Trading Volume Over Time")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Volume")
axes[0].grid(alpha=0.3)

axes[1].plot(
    df["datetime"],
    df["count"],
    linewidth=1
)

axes[1].set_title("Number of Trades Over Time")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Trade Count")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# ROLLING VOLATILITY
# ==========================================

df["volatility_7d"] = df["return_1d"].rolling(7).std()
df["volatility_30d"] = df["return_1d"].rolling(30).std()

plt.figure(figsize=(16, 7))

plt.plot(
    df["datetime"],
    df["volatility_7d"],
    label="7-Day Volatility"
)

plt.plot(
    df["datetime"],
    df["volatility_30d"],
    label="30-Day Volatility"
)

plt.title("BTCUSDT Rolling Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# ROLLING VOLATILITY
# ==========================================

df["volatility_7d"] = df["return_1d"].rolling(7).std()
df["volatility_30d"] = df["return_1d"].rolling(30).std()

plt.figure(figsize=(16, 7))

plt.plot(
    df["datetime"],
    df["volatility_7d"],
    label="7-Day Volatility"
)

plt.plot(
    df["datetime"],
    df["volatility_30d"],
    label="30-Day Volatility"
)

plt.title("BTCUSDT Rolling Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# TARGET CREATION
# ==========================================

# Next-day closing price
df["next_close"] = df["close"].shift(-1)

# Next-day return
df["target_return"] = (
    df["next_close"] / df["close"]
) - 1

# Next-day direction
df["target_direction"] = (
    df["next_close"] > df["close"]
).astype(int)

print("Target columns created.")

display(
    df[
        [
            "datetime",
            "close",
            "next_close",
            "target_return",
            "target_direction"
        ]
    ].head(10)
)

In [ ]:
# ==========================================
# TARGET ANALYSIS
# ==========================================

print("Target Return Statistics:")
display(df["target_return"].describe())

print("\nDirection Distribution:")

direction_counts = df["target_direction"].value_counts()

display(direction_counts)

print("\nDirection Percentage:")

direction_percentage = (
    df["target_direction"]
    .value_counts(normalize=True)
    * 100
)

display(direction_percentage)

In [ ]:
# ==========================================
# CORRELATION ANALYSIS
# ==========================================

numeric_cols = df.select_dtypes(
    include=np.number
).columns

correlation = df[numeric_cols].corr()

plt.figure(figsize=(18, 14))

sns.heatmap(
    correlation,
    cmap="coolwarm",
    center=0,
    linewidths=0.3
)

plt.title("Feature Correlation Heatmap")

plt.show()

In [ ]:
# ==========================================
# REMOVE ROWS WITH INSUFFICIENT HISTORY
# ==========================================

before = len(df)

df_model = df.dropna().copy()

after = len(df_model)

print("Rows before removing NaN:", before)
print("Rows after removing NaN:", after)
print("Rows removed:", before - after)

print("\nRemaining missing values:")
print(df_model.isnull().sum().sum())

# 5. Feature Engineering

In [ ]:
# ==========================================
# FEATURE / TARGET DEFINITION
# ==========================================

# Columns that must NOT be used as predictors
exclude_columns = [
    "id",
    "datetime",
    "next_close",
    "target_return",
    "target_direction"
]

feature_columns = [
    col
    for col in df_model.columns
    if col not in exclude_columns
]

X = df_model[feature_columns]

y_reg = df_model["target_return"]

y_cls = df_model["target_direction"]

print("Number of features:", len(feature_columns))

print("\nFeatures:")
print(feature_columns)

print("\nX shape:", X.shape)
print("Regression target shape:", y_reg.shape)
print("Classification target shape:", y_cls.shape)

# 7. Machine Learning Preparation

In [ ]:
# ==========================================
# CHRONOLOGICAL TRAIN / VALIDATION / TEST
# ==========================================

n = len(df_model)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:validation_end]
X_test = X.iloc[validation_end:]

y_reg_train = y_reg.iloc[:train_end]
y_reg_val = y_reg.iloc[train_end:validation_end]
y_reg_test = y_reg.iloc[validation_end:]

y_cls_train = y_cls.iloc[:train_end]
y_cls_val = y_cls.iloc[train_end:validation_end]
y_cls_test = y_cls.iloc[validation_end:]

print("TRAIN:")
print(X_train.shape)

print("\nVALIDATION:")
print(X_val.shape)

print("\nTEST:")
print(X_test.shape)

print("\nDate Ranges:")

print(
    "Train:",
    df_model["datetime"].iloc[0],
    "to",
    df_model["datetime"].iloc[train_end - 1]
)

print(
    "Validation:",
    df_model["datetime"].iloc[train_end],
    "to",
    df_model["datetime"].iloc[validation_end - 1]
)

print(
    "Test:",
    df_model["datetime"].iloc[validation_end],
    "to",
    df_model["datetime"].iloc[-1]
)

In [ ]:
# ==========================================
# FEATURE SCALING
# ==========================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

print("Scaling completed.")

print("Training shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)
print("Test shape:", X_test_scaled.shape)

# 8. Regression — Next-Day Return Prediction

In [ ]:
# ==========================================
# NAIVE REGRESSION BASELINE
# ==========================================

# Persistence assumption:
# Tomorrow's return is assumed to be 0

baseline_reg_pred = np.zeros(len(y_reg_test))

baseline_mae = mean_absolute_error(
    y_reg_test,
    baseline_reg_pred
)

baseline_mse = mean_squared_error(
    y_reg_test,
    baseline_reg_pred
)

baseline_rmse = np.sqrt(baseline_mse)

baseline_r2 = r2_score(
    y_reg_test,
    baseline_reg_pred
)

print("NAIVE REGRESSION BASELINE")
print("--------------------------------")

print(f"MAE  : {baseline_mae:.6f}")
print(f"MSE  : {baseline_mse:.6f}")
print(f"RMSE : {baseline_rmse:.6f}")
print(f"R²   : {baseline_r2:.6f}")

In [ ]:
# ==========================================
# LINEAR REGRESSION
# ==========================================

linear_model = LinearRegression()

linear_model.fit(
    X_train_scaled,
    y_reg_train
)

linear_val_pred = linear_model.predict(
    X_val_scaled
)

linear_test_pred = linear_model.predict(
    X_test_scaled
)

print("Linear Regression trained.")

In [ ]:
# ==========================================
# RIDGE REGRESSION
# ==========================================

ridge_model = Ridge(
    alpha=1.0
)

ridge_model.fit(
    X_train_scaled,
    y_reg_train
)

ridge_val_pred = ridge_model.predict(
    X_val_scaled
)

ridge_test_pred = ridge_model.predict(
    X_test_scaled
)

print("Ridge Regression trained.")

In [ ]:
# ==========================================
# RANDOM FOREST REGRESSOR
# ==========================================

rf_reg = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_reg.fit(
    X_train,
    y_reg_train
)

rf_val_pred = rf_reg.predict(X_val)

rf_test_pred = rf_reg.predict(X_test)

print("Random Forest Regressor trained.")

In [ ]:
# ==========================================
# GRADIENT BOOSTING REGRESSOR
# ==========================================

gb_reg = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    random_state=RANDOM_STATE
)

gb_reg.fit(
    X_train,
    y_reg_train
)

gb_val_pred = gb_reg.predict(X_val)

gb_test_pred = gb_reg.predict(X_test)

print("Gradient Boosting Regressor trained.")

In [ ]:
# ==========================================
# REGRESSION EVALUATION FUNCTION
# ==========================================

def evaluate_regression(
    model_name,
    y_true,
    predictions
):

    mae = mean_absolute_error(
        y_true,
        predictions
    )

    mse = mean_squared_error(
        y_true,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_true,
        predictions
    )

    # Directional accuracy
    directional_accuracy = np.mean(
        np.sign(y_true) == np.sign(predictions)
    )

    return {
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "Directional Accuracy": directional_accuracy
    }

In [ ]:
# ==========================================
# REGRESSION MODEL COMPARISON
# ==========================================

regression_results = []

# Baseline
regression_results.append(
    evaluate_regression(
        "Naive Baseline",
        y_reg_test,
        baseline_reg_pred
    )
)

# Linear Regression
regression_results.append(
    evaluate_regression(
        "Linear Regression",
        y_reg_test,
        linear_test_pred
    )
)

# Ridge
regression_results.append(
    evaluate_regression(
        "Ridge Regression",
        y_reg_test,
        ridge_test_pred
    )
)

# Random Forest
regression_results.append(
    evaluate_regression(
        "Random Forest",
        y_reg_test,
        rf_test_pred
    )
)

# Gradient Boosting
regression_results.append(
    evaluate_regression(
        "Gradient Boosting",
        y_reg_test,
        gb_test_pred
    )
)

regression_results_df = pd.DataFrame(
    regression_results
)

regression_results_df = regression_results_df.sort_values(
    "RMSE"
).reset_index(drop=True)

display(
    regression_results_df.style.format({
        "MAE": "{:.6f}",
        "MSE": "{:.6f}",
        "RMSE": "{:.6f}",
        "R2": "{:.4f}",
        "Directional Accuracy": "{:.2%}"
    })
)

In [ ]:
# ==========================================
# REGRESSION MODEL COMPARISON
# ==========================================

plt.figure(figsize=(12, 6))

sns.barplot(
    data=regression_results_df,
    x="RMSE",
    y="Model"
)

plt.title("Regression Model Comparison — RMSE")
plt.xlabel("RMSE")
plt.ylabel("Model")

plt.grid(axis="x", alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# ACTUAL VS PREDICTED RETURNS
# ==========================================

plt.figure(figsize=(16, 6))

plt.plot(
    y_reg_test.index,
    y_reg_test.values,
    label="Actual Return",
    linewidth=1
)

plt.plot(
    y_reg_test.index,
    gb_test_pred,
    label="Predicted Return",
    linewidth=1
)

plt.title("Actual vs Predicted BTCUSDT Next-Day Returns")
plt.xlabel("Test Observation")
plt.ylabel("Next-Day Return")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# REGRESSION RESIDUAL ANALYSIS
# ==========================================

residuals = y_reg_test.values - gb_test_pred

plt.figure(figsize=(12, 6))

sns.histplot(
    residuals,
    bins=60,
    kde=True
)

plt.title("Prediction Error Distribution")
plt.xlabel("Residual")
plt.ylabel("Frequency")

plt.grid(alpha=0.2)

plt.show()

# 9. Classification — Market Direction Prediction

In [ ]:
# ==========================================
# CLASSIFICATION BASELINE
# ==========================================

# Baseline predicts the majority class
majority_class = y_cls_train.mode()[0]

baseline_cls_pred = np.full(
    len(y_cls_test),
    majority_class
)

print("Majority class:", majority_class)

print(
    "Baseline Accuracy:",
    accuracy_score(
        y_cls_test,
        baseline_cls_pred
    )
)

In [ ]:
# ==========================================
# LOGISTIC REGRESSION
# ==========================================

logistic_model = LogisticRegression(
    max_iter=2000,
    random_state=RANDOM_STATE
)

logistic_model.fit(
    X_train_scaled,
    y_cls_train
)

logistic_val_pred = logistic_model.predict(
    X_val_scaled
)

logistic_test_pred = logistic_model.predict(
    X_test_scaled
)

logistic_test_proba = logistic_model.predict_proba(
    X_test_scaled
)[:, 1]

print("Logistic Regression trained.")

In [ ]:
# ==========================================
# RANDOM FOREST CLASSIFIER
# ==========================================

rf_cls = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_cls.fit(
    X_train,
    y_cls_train
)

rf_cls_val_pred = rf_cls.predict(
    X_val
)

rf_cls_test_pred = rf_cls.predict(
    X_test
)

rf_cls_test_proba = rf_cls.predict_proba(
    X_test
)[:, 1]

print("Random Forest Classifier trained.")

In [ ]:
# ==========================================
# GRADIENT BOOSTING CLASSIFIER
# ==========================================

gb_cls = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    random_state=RANDOM_STATE
)

gb_cls.fit(
    X_train,
    y_cls_train
)

gb_cls_val_pred = gb_cls.predict(
    X_val
)

gb_cls_test_pred = gb_cls.predict(
    X_test
)

gb_cls_test_proba = gb_cls.predict_proba(
    X_test
)[:, 1]

print("Gradient Boosting Classifier trained.")

In [ ]:
# ==========================================
# CLASSIFICATION EVALUATION FUNCTION
# ==========================================

def evaluate_classification(
    model_name,
    y_true,
    predictions,
    probabilities
):

    accuracy = accuracy_score(
        y_true,
        predictions
    )

    precision = precision_score(
        y_true,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        predictions,
        zero_division=0
    )

    auc = roc_auc_score(
        y_true,
        probabilities
    )

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": auc
    }

In [ ]:
# ==========================================
# CLASSIFICATION MODEL COMPARISON
# ==========================================

classification_results = []

# Baseline
baseline_probability = np.full(
    len(y_cls_test),
    majority_class
)

classification_results.append(
    evaluate_classification(
        "Majority Baseline",
        y_cls_test,
        baseline_cls_pred,
        baseline_probability
    )
)

# Logistic Regression
classification_results.append(
    evaluate_classification(
        "Logistic Regression",
        y_cls_test,
        logistic_test_pred,
        logistic_test_proba
    )
)

# Random Forest
classification_results.append(
    evaluate_classification(
        "Random Forest",
        y_cls_test,
        rf_cls_test_pred,
        rf_cls_test_proba
    )
)

# Gradient Boosting
classification_results.append(
    evaluate_classification(
        "Gradient Boosting",
        y_cls_test,
        gb_cls_test_pred,
        gb_cls_test_proba
    )
)

classification_results_df = pd.DataFrame(
    classification_results
)

classification_results_df = classification_results_df.sort_values(
    "F1",
    ascending=False
).reset_index(drop=True)

display(
    classification_results_df.style.format({
        "Accuracy": "{:.2%}",
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1": "{:.2%}",
        "ROC-AUC": "{:.4f}"
    })
)

In [ ]:
# ==========================================
# CONFUSION MATRIX
# ==========================================

# Use Gradient Boosting Classifier
cm = confusion_matrix(
    y_cls_test,
    gb_cls_test_pred
)

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Down / Same", "Up"],
    yticklabels=["Down / Same", "Up"]
)

plt.title("Gradient Boosting — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

print("Classification Report:")
print(
    classification_report(
        y_cls_test,
        gb_cls_test_pred,
        target_names=["Down / Same", "Up"],
        zero_division=0
    )
)

In [ ]:
# ==========================================
# ROC CURVE
# ==========================================

fpr, tpr, _ = roc_curve(
    y_cls_test,
    gb_cls_test_proba
)

auc_score = roc_auc_score(
    y_cls_test,
    gb_cls_test_proba
)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f"Gradient Boosting (AUC = {auc_score:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# CLASSIFICATION MODEL COMPARISON PLOT
# ==========================================

plt.figure(figsize=(12, 6))

sns.barplot(
    data=classification_results_df,
    x="F1",
    y="Model"
)

plt.title("Classification Model Comparison — F1 Score")
plt.xlabel("F1 Score")
plt.ylabel("Model")

plt.grid(axis="x", alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# RANDOM FOREST REGRESSION FEATURE IMPORTANCE
# ==========================================

rf_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": rf_reg.feature_importances_
})

rf_importance = rf_importance.sort_values(
    "Importance",
    ascending=False
)

display(rf_importance.head(20))

# 10. Feature Importance

In [ ]:
# ==========================================
# FEATURE IMPORTANCE PLOT
# ==========================================

top_features = rf_importance.head(20)

plt.figure(figsize=(12, 8))

sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature"
)

plt.title("Top 20 Features — Random Forest Regression")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.grid(axis="x", alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# PERMUTATION IMPORTANCE
# ==========================================

perm_result = permutation_importance(
    rf_reg,
    X_test,
    y_reg_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

permutation_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": perm_result.importances_mean,
    "Std": perm_result.importances_std
})

permutation_df = permutation_df.sort_values(
    "Importance",
    ascending=False
)

display(
    permutation_df.head(20)
)

In [ ]:
# ==========================================
# CLASSIFICATION FEATURE IMPORTANCE
# ==========================================

classification_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": gb_cls.feature_importances_
})

classification_importance = (
    classification_importance
    .sort_values(
        "Importance",
        ascending=False
    )
)

display(
    classification_importance.head(20)
)

# 11. Model Selection

In [ ]:
# ==========================================
# VALIDATION-BASED REGRESSION MODEL SELECTION
# ==========================================

validation_predictions = {
    "Linear Regression": linear_val_pred,
    "Ridge Regression": ridge_val_pred,
    "Random Forest": rf_val_pred,
    "Gradient Boosting": gb_val_pred
}

validation_results = []

for model_name, predictions in validation_predictions.items():

    result = evaluate_regression(
        model_name,
        y_reg_val,
        predictions
    )

    validation_results.append(result)

validation_results_df = pd.DataFrame(
    validation_results
)

validation_results_df = validation_results_df.sort_values(
    "RMSE"
).reset_index(drop=True)

print("Validation Results:")
display(
    validation_results_df.style.format({
        "MAE": "{:.6f}",
        "MSE": "{:.6f}",
        "RMSE": "{:.6f}",
        "R2": "{:.4f}",
        "Directional Accuracy": "{:.2%}"
    })
)

best_regression_name = (
    validation_results_df.iloc[0]["Model"]
)

print(
    "\nSelected regression model:",
    best_regression_name
)

In [ ]:
# ==========================================
# VALIDATION-BASED CLASSIFICATION SELECTION
# ==========================================

validation_cls_predictions = {
    "Logistic Regression": (
        logistic_val_pred,
        logistic_model.predict_proba(X_val_scaled)[:, 1]
    ),
    
    "Random Forest": (
        rf_cls_val_pred,
        rf_cls.predict_proba(X_val)[:, 1]
    ),
    
    "Gradient Boosting": (
        gb_cls_val_pred,
        gb_cls.predict_proba(X_val)[:, 1]
    )
}

validation_cls_results = []

for model_name, (predictions, probabilities) in validation_cls_predictions.items():

    result = evaluate_classification(
        model_name,
        y_cls_val,
        predictions,
        probabilities
    )

    validation_cls_results.append(result)

validation_cls_results_df = pd.DataFrame(
    validation_cls_results
)

validation_cls_results_df = (
    validation_cls_results_df
    .sort_values(
        "F1",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Validation Classification Results:")

display(
    validation_cls_results_df.style.format({
        "Accuracy": "{:.2%}",
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1": "{:.2%}",
        "ROC-AUC": "{:.4f}"
    })
)

best_classification_name = (
    validation_cls_results_df.iloc[0]["Model"]
)

print(
    "\nSelected classification model:",
    best_classification_name
)

# 12. Final Model Evaluation

In [ ]:
# ==========================================
# FINAL REGRESSION TEST RESULT
# ==========================================

regression_model_predictions = {
    "Linear Regression": linear_test_pred,
    "Ridge Regression": ridge_test_pred,
    "Random Forest": rf_test_pred,
    "Gradient Boosting": gb_test_pred
}

best_regression_test_pred = (
    regression_model_predictions[
        best_regression_name
    ]
)

final_regression_result = evaluate_regression(
    best_regression_name,
    y_reg_test,
    best_regression_test_pred
)

print("FINAL REGRESSION TEST PERFORMANCE")
print("-----------------------------------")

for key, value in final_regression_result.items():

    if key == "Model":
        print(f"{key}: {value}")

    elif key == "Directional Accuracy":
        print(f"{key}: {value:.2%}")

    else:
        print(f"{key}: {value:.6f}")

In [ ]:
# ==========================================
# FINAL CLASSIFICATION TEST RESULT
# ==========================================

classification_predictions = {
    "Logistic Regression": (
        logistic_test_pred,
        logistic_test_proba
    ),
    
    "Random Forest": (
        rf_cls_test_pred,
        rf_cls_test_proba
    ),
    
    "Gradient Boosting": (
        gb_cls_test_pred,
        gb_cls_test_proba
    )
}

best_cls_test_pred, best_cls_test_proba = (
    classification_predictions[
        best_classification_name
    ]
)

final_classification_result = evaluate_classification(
    best_classification_name,
    y_cls_test,
    best_cls_test_pred,
    best_cls_test_proba
)

print("FINAL CLASSIFICATION TEST PERFORMANCE")
print("--------------------------------------")

for key, value in final_classification_result.items():

    if key == "Model":
        print(f"{key}: {value}")
    else:
        print(f"{key}: {value:.2%}" if key != "ROC-AUC"
              else f"{key}: {value:.4f}")

# 13. Next-Day Price Prediction

In [ ]:
# ==========================================
# ESTIMATED NEXT-DAY PRICE
# ==========================================

test_dates = df_model.loc[
    y_reg_test.index,
    "datetime"
].values

current_close_test = df_model.loc[
    y_reg_test.index,
    "close"
].values

actual_next_close = (
    current_close_test *
    (1 + y_reg_test.values)
)

predicted_next_close = (
    current_close_test *
    (1 + best_regression_test_pred)
)

prediction_df = pd.DataFrame({
    "datetime": test_dates,
    "current_close": current_close_test,
    "actual_next_close": actual_next_close,
    "predicted_return": best_regression_test_pred,
    "predicted_next_close": predicted_next_close,
    "actual_direction": y_cls_test.values
})

display(prediction_df.head(20))

In [ ]:
# ==========================================
# ACTUAL VS PREDICTED NEXT-DAY PRICE
# ==========================================

plt.figure(figsize=(16, 7))

plt.plot(
    prediction_df["datetime"],
    prediction_df["actual_next_close"],
    label="Actual Next-Day Close",
    linewidth=1.5
)

plt.plot(
    prediction_df["datetime"],
    prediction_df["predicted_next_close"],
    label="Predicted Next-Day Close",
    linewidth=1.2
)

plt.title(
    f"{best_regression_name} — Actual vs Predicted Next-Day BTCUSDT Price"
)

plt.xlabel("Date")
plt.ylabel("BTCUSDT Price")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

# 14. Latest BTCUSDT Prediction

In [ ]:
# ==========================================
# NEXT-DAY PREDICTION USING LATEST DATA
# ==========================================

latest_features = X.iloc[[-1]]

latest_date = df_model["datetime"].iloc[-1]

latest_close = df_model["close"].iloc[-1]

# Regression prediction
if best_regression_name == "Linear Regression":
    latest_regression_pred = linear_model.predict(
        scaler.transform(latest_features)
    )[0]

elif best_regression_name == "Ridge Regression":
    latest_regression_pred = ridge_model.predict(
        scaler.transform(latest_features)
    )[0]

elif best_regression_name == "Random Forest":
    latest_regression_pred = rf_reg.predict(
        latest_features
    )[0]

elif best_regression_name == "Gradient Boosting":
    latest_regression_pred = gb_reg.predict(
        latest_features
    )[0]


estimated_next_close = (
    latest_close *
    (1 + latest_regression_pred)
)

# Classification prediction
if best_classification_name == "Logistic Regression":

    latest_direction = logistic_model.predict(
        scaler.transform(latest_features)
    )[0]

    latest_probability = logistic_model.predict_proba(
        scaler.transform(latest_features)
    )[0, 1]

elif best_classification_name == "Random Forest":

    latest_direction = rf_cls.predict(
        latest_features
    )[0]

    latest_probability = rf_cls.predict_proba(
        latest_features
    )[0, 1]

else:

    latest_direction = gb_cls.predict(
        latest_features
    )[0]

    latest_probability = gb_cls.predict_proba(
        latest_features
    )[0, 1]


print("==========================================")
print("LATEST BTCUSDT PREDICTION")
print("==========================================")

print(f"Latest Date: {latest_date}")
print(f"Current Close: {latest_close:.2f}")

print(
    f"\nPredicted Next-Day Return: "
    f"{latest_regression_pred:.4%}"
)

print(
    f"Estimated Next-Day Close: "
    f"{estimated_next_close:.2f}"
)

print(
    f"\nPredicted Direction: "
    f"{'UP' if latest_direction == 1 else 'DOWN / SAME'}"
)

print(
    f"Probability of UP: "
    f"{latest_probability:.2%}"
)

print("\n==========================================")
print("IMPORTANT: Experimental prediction only.")
print("Not financial advice or a guaranteed outcome.")
print("==========================================")

# 15. Save Models & Results

In [ ]:
# ==========================================
# SAVE MODELS
# ==========================================

# Save regression models
joblib.dump(
    rf_reg,
    "btc_rf_regressor.pkl"
)

joblib.dump(
    gb_reg,
    "btc_gradient_boosting_regressor.pkl"
)

# Save classification models
joblib.dump(
    rf_cls,
    "btc_rf_classifier.pkl"
)

joblib.dump(
    gb_cls,
    "btc_gradient_boosting_classifier.pkl"
)

# Save scaler
joblib.dump(
    scaler,
    "btc_feature_scaler.pkl"
)

# Save feature list
joblib.dump(
    feature_columns,
    "btc_feature_columns.pkl"
)

print("Models and preprocessing objects saved successfully.")

In [ ]:
# ==========================================
# EXPORT PREDICTION RESULTS
# ==========================================

prediction_df.to_csv(
    "btc_prediction_results.csv",
    index=False
)

regression_results_df.to_csv(
    "regression_model_comparison.csv",
    index=False
)

classification_results_df.to_csv(
    "classification_model_comparison.csv",
    index=False
)

print("Prediction and model comparison files saved.")

# Final Project Summary

## Dataset

The project uses historical BTCUSDT daily OHLCV data containing:

- Open
- High
- Low
- Close
- Amount
- Volume
- Trade Count
- Timestamp

## Machine Learning Tasks

### Regression

The primary regression target is:

`Next-Day Return`

The project evaluates:

- MAE
- MSE
- RMSE
- R²
- Directional Accuracy

### Classification

The secondary classification target is:

- `1` → Next-day price increases
- `0` → Next-day price does not increase

The classification models are evaluated using:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion Matrix

## Models

The project compares:

### Regression

1. Naive Baseline
2. Linear Regression
3. Ridge Regression
4. Random Forest Regressor
5. Gradient Boosting Regressor

### Classification

1. Majority Class Baseline
2. Logistic Regression
3. Random Forest Classifier
4. Gradient Boosting Classifier

## Important Methodology

The project uses chronological train/validation/test splitting rather than random splitting.

This is important because financial time-series data has a temporal relationship and random shuffling can introduce information leakage.

## Conclusion

The final model should only be considered successful if it demonstrates useful performance on the unseen chronological test period and improves meaningfully over the naive baseline.

Historical performance does not guarantee future performance because cryptocurrency markets are non-stationary and affected by many external factors.

> This project is an educational machine-learning experiment and should not be interpreted as financial advice or a guaranteed trading strategy.